TF-IDF IMPLEMENTATION OF AI PREFERENCE CLASSIFIER

In [ ]:
!pip install sentencepiece
!pip install scikit-learn
!pip install tensorflow
!pip install torch

In [1]:
import sentencepiece as spm

In [ ]:
#Training the tokenizer model
spm.SentencePieceTrainer.Train(
    '--input=dataset/train.csv --model_prefix=vocab --vocab_size=2000 --model_type=bpe'
)

In [100]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

# import tensorflow as tf
# from tensorflow import keras
# from tensorflow.keras import layers

import torch
import torch.nn as nn
import torch.optim as optim

import random

In [3]:
#Loading the trained model
sp = spm.SentencePieceProcessor()
sp.load('vocab.model')

True

In [92]:
dataset = pd.read_csv('dataset/train.csv')
dataset = dataset.dropna(subset=['prompt', 'response_a', 'response_b'])

In [90]:
dataset.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1.0,0.0,0.0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0.0,1.0,0.0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0.0,0.0,1.0
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1.0,0.0,0.0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0.0,1.0,0.0


In [63]:
prompt = dataset.iloc[:]['prompt']
answer1 = dataset.iloc[:]['response_a']
answer2 = dataset.iloc[:]['response_b']

In [64]:
token1 = sp.encode(prompt, out_type=int)
token2 = sp.encode(answer1, out_type=int)
token3 = sp.encode(answer2, out_type=int)

print(token1)
print(token2)
print(token3)

TypeError: not a string

In [8]:
print(len(token1))
print(len(token2))
print(len(token3))

62
1604
396


In [9]:
vocab = [str(i) for i in range(sp.get_piece_size())]
vectorizer = TfidfVectorizer(vocabulary=vocab)

In [103]:
class TFIDFdata():
    def __init__(self, data, test_size = 0.3):

        #Extracting the input
        self.prompt = data.iloc[:]['prompt']
        self.answer1 = data.iloc[:]['response_a']
        self.answer2 = data.iloc[:]['response_b']

        self.input = []

        for i in range(len(data)):
            self.input.append(vectorizer.fit_transform([self.prompt[i],self.answer1[i],self.answer2[i]]).toarray().flatten())

        #input and target values
        self.input = torch.tensor(np.array(self.input),dtype=torch.float32)
        self.target = torch.tensor(dataset.iloc[: ,-3:].astype(float).values,dtype=torch.float32)

        self.num_train = int(np.shape(data)[0] * (1-test_size))
        self.num_val = np.shape(dataset)[0] - self.num_train

    def get_dataloader(self,train):
        if train:
            self.id = list(range(0, self.num_train))
            random.shuffle(self.id)
        else:
            self.id = list(range(self.num_train, self.num_train + self.num_val))


In [96]:
data = TFIDFdata(dataset)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [97]:
np.shape(data.input)

(30000, 6000)

In [27]:
input1 = vectorizer.fit_transform([prompt,answer1,answer2]).toarray()
input1 = input1.flatten()
input1

array([0., 0., 0., ..., 0., 0., 0.], shape=(6000,))

In [53]:
output1 = torch.tensor(dataset.iloc[0 ,-3:].astype(float).values,dtype=torch.float32)
print(output1)

tensor([1., 0., 0.])


ADDING THE FEED FORWARD NETWORK

In [54]:
X = torch.tensor(input1, dtype=torch.float32) 
y = output1

In [55]:
y.shape

torch.Size([3])

In [60]:
class FFN(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
            nn.Sigmoid()           
        )
    def forward(self,x):
        return self.net(x)

In [61]:
model = FFN(len(input1))
optimizer = optim.Adam(model.parameters())
criterion = nn.BCELoss()

In [62]:
for epoch in range(1000):
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()

    print(outputs)
    print(f"Epoch {epoch}: Loss = {loss.item():.4f}")

tensor([0.5154, 0.5043, 0.5113], grad_fn=<SigmoidBackward0>)
Epoch 0: Loss = 0.6936
tensor([0.5169, 0.5028, 0.5097], grad_fn=<SigmoidBackward0>)
Epoch 1: Loss = 0.6904
tensor([0.5182, 0.5013, 0.5080], grad_fn=<SigmoidBackward0>)
Epoch 2: Loss = 0.6875
tensor([0.5195, 0.4995, 0.5063], grad_fn=<SigmoidBackward0>)
Epoch 3: Loss = 0.6843
tensor([0.5208, 0.4980, 0.5047], grad_fn=<SigmoidBackward0>)
Epoch 4: Loss = 0.6814
tensor([0.5222, 0.4965, 0.5030], grad_fn=<SigmoidBackward0>)
Epoch 5: Loss = 0.6783
tensor([0.5235, 0.4946, 0.5013], grad_fn=<SigmoidBackward0>)
Epoch 6: Loss = 0.6751
tensor([0.5248, 0.4930, 0.4998], grad_fn=<SigmoidBackward0>)
Epoch 7: Loss = 0.6722
tensor([0.5260, 0.4915, 0.4979], grad_fn=<SigmoidBackward0>)
Epoch 8: Loss = 0.6692
tensor([0.5277, 0.4897, 0.4961], grad_fn=<SigmoidBackward0>)
Epoch 9: Loss = 0.6658
tensor([0.5289, 0.4877, 0.4941], grad_fn=<SigmoidBackward0>)
Epoch 10: Loss = 0.6624
tensor([0.5301, 0.4860, 0.4928], grad_fn=<SigmoidBackward0>)
Epoch 11: Loss